In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
%%sql


In [2]:
data = pd.read_csv("reddit_posts.csv")

In [11]:
import json

n = 3  # Minimum number of stored top-level comments; adjust as needed.

# Count the comments actually available, excluding nested replies.
def count_stored_comments(value):
    if pd.isna(value) or not value.strip():
        return 0
    comments = json.loads(value)
    return len(comments) if isinstance(comments, list) else 0

data["stored_comment_count"] = data["comments"].apply(count_stored_comments)
data = data.loc[data["stored_comment_count"] >= n].reset_index(drop=True)

data.head()


,post_id,url,user_posted,title,description,num_comments,date_posted,community_name,num_upvotes,photos,...,author_icon,user_id,archived,community_rules,weekly_visitors,weekly_contributions,community_created_date,community_type,locked_post,stored_comment_count
0,t3_1vdjosh,https://www.reddit.com/r/UKPersonalFinance/com...,ColdBeerHero,Does anyone have experience of getting a mortg...,I have a mortgage which I entered years ago. S...,38,2026-08-02T14:35:52.740Z,UKPersonalFinance,19,NaN,...,https://www.redditstatic.com/avatars/defaults/...,t2_g1dndcw4,False,"[{""description"":""Be kind. Don't judge people's...",650043.0,9799.0,2013-03-08T13:00:52.001Z,Public,NaN,14
1,t3_1o6agbx,https://www.reddit.com/r/UKJobs/comments/1o6ag...,Cultural-Badger-6032,"Colour me shocked, UK unemployment is at 4 yea...",NaN,941,2025-10-14T09:05:27.453Z,UKJobs,1297,"[""https://i.redd.it/a9pfbxabm1vf1.png""]",...,https://styles.redditmedia.com/t5_d9va7f/style...,t2_1gfvxyho3u,True,"[{""description"":""Conversations about a job lis...",403733.0,8614.0,2011-03-27T17:54:40.234Z,Public,NaN,17
2,t3_1ijyw7g,https://reddit.com/r/sanfrancisco/comments/1ij...,trappedmaps,to people that live on a busy street (Franklin...,I moved to a busy street 3 months ago after li...,196,2025-02-07T16:24:04.532Z,sanfrancisco,105,NaN,...,https://styles.redditmedia.com/t5_yfhje/styles...,t2_23xoehih,True,NaN,NaN,NaN,NaN,NaN,NaN,24
3,t3_1to9ron,https://www.reddit.com/r/unpopularopinion/comm...,Former-Pattern5250,Putting anyone on a pedestal is simply a sign ...,If you actually defend and fight over a person...,79,2026-05-26T15:08:01.619Z,unpopularopinion,51,NaN,...,https://www.redditstatic.com/avatars/defaults/...,t2_2d0ggk6sgi,False,NaN,NaN,NaN,NaN,NaN,NaN,12
4,t3_1r3xdur,https://www.reddit.com/r/OaklandFood/comments/...,Equivalent-Fact3939,Sizzling rice claypots,"Good people of Oakland, know where I can get o...",12,2026-02-13T18:42:31.635Z,OaklandFood,19,NaN,...,https://preview.redd.it/snoovatar/avatars/5502...,t2_mhtucgbg,False,NaN,NaN,NaN,NaN,NaN,NaN,6


In [12]:
# Export the filtered posts: one row per post, three columns.
import json

MISSING_TEXT = {"[removed]", "[deleted]", "null", "nan", "none"}

def has_usable_text(value):
    return isinstance(value, str) and bool(value.strip()) and value.strip().lower() not in MISSING_TEXT

def extract_comment_texts(value):
    if pd.isna(value) or not value.strip():
        return []
    records = json.loads(value)
    if not isinstance(records, list):
        return []

    collected = []

    def collect(items):
        for item in items:
            text = item.get("comment", item.get("reply", ""))
            timestamp = item.get("date_of_comment", item.get("date_of_reply"))
            if has_usable_text(text):
                collected.append((timestamp, text))
            collect(item.get("replies") or [])

    collect(records)
    # ISO UTC timestamps sort chronologically; missing dates go last.
    collected.sort(key=lambda item: (not bool(item[0]), item[0] or ""))
    return [text for _, text in collected]

# Enforce the minimum again at export time.
filtered_data = data.loc[data["comments"].apply(count_stored_comments) >= n]
export_data = filtered_data[["title", "description"]].copy()
export_data["comments"] = filtered_data["comments"].apply(extract_comment_texts)
# Require complete exported fields and at least n usable comments after cleaning.
complete = (
    export_data["title"].apply(has_usable_text)
    & export_data["description"].apply(has_usable_text)
    & export_data["comments"].apply(lambda comments: len(comments) >= n)
)
export_data = export_data.loc[complete].reset_index(drop=True)
export_data["comments"] = export_data["comments"].apply(
    lambda comments: json.dumps(comments, ensure_ascii=False)
)
export_data.to_csv("reddit_text_only.csv", index=False, encoding="utf-8")
export_data.head()


,title,description,comments
0,Does anyone have experience of getting a mortg...,I have a mortgage which I entered years ago. S...,"[""Why would you run the risk? Not clever."", ""W..."
1,to people that live on a busy street (Franklin...,I moved to a busy street 3 months ago after li...,"[""For me it feels like ambient noise, and I ac..."
2,Putting anyone on a pedestal is simply a sign ...,If you actually defend and fight over a person...,"[""Generally yes, but it's probably not the wor..."
3,Sizzling rice claypots,"Good people of Oakland, know where I can get o...","[""Best Taste Restaurant in Chinatown. I think ..."
4,Mother wants me to start paying her for rent.,"So, I’m a 20 year old male and my mother start...","[""We made it clear to our kids that if they ar..."


In [13]:
export_data.shape

(188140, 3)